# 01 - Load Data into MySQL

This notebook loads the Olist CSV files into a MySQL database (`olist`) so they can be queried with SQL.

In [2]:
import os
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Password is typed at runtime, so it is never saved in this file
password = getpass.getpass('Enter MySQL password: ')

url = URL.create(
    drivername='mysql+pymysql',
    username='root',
    password=password,
    host='127.0.0.1',
    port=3306,
    database='olist',
)
engine = create_engine(url)

files = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

date_cols = {
    'orders': ['order_purchase_timestamp', 'order_approved_at',
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date'],
}

for table, file in files.items():
    df = pd.read_csv(os.path.join('..', 'data', file))
    for c in date_cols.get(table, []):
        # Dates in these CSVs are in DD-MM-YYYY HH:MM format
        df[c] = pd.to_datetime(df[c], format='%d-%m-%Y %H:%M', errors='coerce')
    df.to_sql(table, engine, if_exists='replace', index=False, chunksize=10000)
    print(table, df.shape)

Enter MySQL password:  ········


orders (99441, 8)
order_items (112650, 7)
order_payments (103886, 5)
customers (99441, 5)
products (32951, 9)


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xad in position 715: invalid start byte